In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

In [2]:
df = pd.read_csv("../data/Loan_approval.csv")

df.head()

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,4822,Female,No,2,Graduate,Yes,48898,0,176,120.0,0.0,Urban,N
1,2364,Female,Yes,1,Not Graduate,Yes,29161,1061,172,240.0,0.0,Rural,N
2,3729,Female,No,2,Graduate,Yes,16279,2679,162,180.0,0.0,Semiurban,Y
3,624,Male,Yes,3+,Grad,No,46572,4801,-50,240.0,0.0,Urban,N
4,1074,Female,No,3+,Not Graduate,Yes,26869,14407,114,360.0,0.0,Semiurban,N


In [3]:
print("Shape:", df.shape)

df.info()

Shape: (5000, 13)
<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Loan_ID            5000 non-null   int64  
 1   Gender             4923 non-null   str    
 2   Married            4934 non-null   str    
 3   Dependents         4925 non-null   str    
 4   Education          5000 non-null   str    
 5   Self_Employed      4902 non-null   str    
 6   ApplicantIncome    5000 non-null   int64  
 7   CoapplicantIncome  5000 non-null   int64  
 8   LoanAmount         5000 non-null   int64  
 9   Loan_Amount_Term   4938 non-null   float64
 10  Credit_History     4926 non-null   float64
 11  Property_Area      4936 non-null   str    
 12  Loan_Status        5000 non-null   str    
dtypes: float64(2), int64(4), str(7)
memory usage: 648.8 KB


In [4]:
df.isnull().sum()


Loan_ID               0
Gender               77
Married              66
Dependents           75
Education             0
Self_Employed        98
ApplicantIncome       0
CoapplicantIncome     0
LoanAmount            0
Loan_Amount_Term     62
Credit_History       74
Property_Area        64
Loan_Status           0
dtype: int64

In [6]:
for col in[
    "Gender",
    "Married",
    "Dependents",
    "Self_Employed",
    "Property_Area"
]:
    print("\n", col)
    print(df[col].value_counts(dropna=False))


 Gender
Gender
Female    2425
Male      2335
Fmale       84
Mle         79
NaN         77
Name: count, dtype: int64

 Married
Married
Yes    2417
No     2378
Y        75
NaN      66
N        64
Name: count, dtype: int64

 Dependents
Dependents
1       1220
3+      1218
0       1201
2       1195
four      91
NaN       75
Name: count, dtype: int64

 Self_Employed
Self_Employed
No      2445
Yes     2357
Self     100
NaN       98
Name: count, dtype: int64

 Property_Area
Property_Area
Urban         1629
Rural         1612
Semiurban     1565
semi-urban      66
Metro           64
NaN             64
Name: count, dtype: int64


In [7]:
df["Gender"]=df["Gender"].replace({
    "Mle":"Male",
    "Fmale":"Female"
})

In [8]:
df["Married"]=df["Married"].replace({
    "Y":"Yes",
    "N": "No"
})

In [9]:
df["Dependents"]=df["Dependents"].replace({
    "four":4
})

In [11]:
print(df["Self_Employed"].value_counts(dropna=False))

print()

print(df["Property_Area"].value_counts(dropna=False))

Self_Employed
No      2445
Yes     2357
Self     100
NaN       98
Name: count, dtype: int64

Property_Area
Urban         1629
Rural         1612
Semiurban     1565
semi-urban      66
Metro           64
NaN             64
Name: count, dtype: int64


In [12]:
df["Self_Employed"]=df["Self_Employed"].replace({
    "Self":"Yes"
})

In [13]:
df["Property_Area"]=df["Property_Area"].replace({
    "semi-urban":"Semiurban",
    "Metro":"Urban"
})

In [15]:
for col in[
    "Gender",
    "Married",
    "Dependents",
    "Self_Employed",
    "Property_Area"
]:
    print("\n", col)

    print(df[col].value_counts(dropna=False))


 Gender
Gender
Female    2509
Male      2414
NaN         77
Name: count, dtype: int64

 Married
Married
Yes    2492
No     2442
NaN      66
Name: count, dtype: int64

 Dependents
Dependents
1      1220
3+     1218
0      1201
2      1195
4        91
NaN      75
Name: count, dtype: int64

 Self_Employed
Self_Employed
Yes    2457
No     2445
NaN      98
Name: count, dtype: int64

 Property_Area
Property_Area
Urban        1693
Semiurban    1631
Rural        1612
NaN            64
Name: count, dtype: int64


In [16]:
df.describe()

,Loan_ID,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History
count,5000.00000,5000.000000,5000.000000,5000.000000,4938.000000,4926.000000
mean,2505.14020,24802.145400,2006.445200,213.173200,235.532199,0.494925
std,1441.64834,13185.277557,3333.975591,747.150068,122.680260,0.565573
min,1.00000,3955.000000,0.000000,-50.000000,120.000000,-1.000000
25%,1254.75000,15537.000000,0.000000,115.000000,180.000000,0.000000
50%,2508.50000,21840.000000,0.000000,148.000000,240.000000,0.000000
75%,3752.25000,30762.500000,2911.500000,184.000000,360.000000,1.000000
max,5000.00000,113002.000000,37830.000000,9999.000000,999.000000,2.000000


In [17]:
df["Credit_History"].value_counts(dropna=False)

Credit_History
 0.0    2408
 1.0    2346
 2.0      88
-1.0      84
 NaN      74
Name: count, dtype: int64

In [18]:
df["Credit_History"] = df["Credit_History"].replace({
    2: np.nan,
    -1: np.nan
})

In [19]:
print(df["LoanAmount"].sort_values().head(20))

4580   -50
668    -50
3182   -50
2230   -50
2236   -50
106    -50
1164   -50
1178   -50
3735   -50
4926   -50
2839   -50
3928   -50
4698   -50
4609   -50
3881   -50
4468   -50
4888   -50
2318   -50
4078   -50
1914   -50
Name: LoanAmount, dtype: int64


In [20]:
print(
    df["LoanAmount"]
    .sort_values(ascending=False)
    .head(20)
)

856     9999
3131    9999
1064    9999
1065    9999
3087    9999
1185    9999
793     9999
4362    9999
1704    9999
2417    9999
1655    9999
3937    9999
4946    9999
2539    9999
3889    9999
1613    9999
4736    9999
3902    9999
3823    9999
3860    9999
Name: LoanAmount, dtype: int64


In [21]:
df["LoanAmount"] = df["LoanAmount"].replace(
    {
        -50: np.nan,
        9999: np.nan
    }
)

In [22]:
df["Loan_Amount_Term"].value_counts(
    dropna=False
)

Loan_Amount_Term
240.0    1279
180.0    1214
360.0    1207
120.0    1176
999.0      62
NaN        62
Name: count, dtype: int64

In [23]:
df["Loan_Amount_Term"] = df["Loan_Amount_Term"].replace(
    999,
    np.nan
)

In [24]:
print(df["Credit_History"].value_counts(dropna=False))

print(df["LoanAmount"].describe())

print(df["Loan_Amount_Term"].value_counts(dropna=False))

Credit_History
0.0    2408
1.0    2346
NaN     246
Name: count, dtype: int64
count    4937.000000
mean      159.538991
std       136.993130
min       -27.000000
25%       116.000000
50%       148.000000
75%       183.000000
max      1500.000000
Name: LoanAmount, dtype: float64
Loan_Amount_Term
240.0    1279
180.0    1214
360.0    1207
120.0    1176
NaN       124
Name: count, dtype: int64


In [25]:
df[df["LoanAmount"] < 0]["LoanAmount"].value_counts()

LoanAmount
-5.0     1
-12.0    1
-25.0    1
-27.0    1
Name: count, dtype: int64

In [26]:
df.loc[
    df["LoanAmount"] < 0,
    "LoanAmount"
] = np.nan

In [27]:
df[df["LoanAmount"] > 1000]["LoanAmount"].value_counts()

LoanAmount
1500.0    44
Name: count, dtype: int64

In [28]:
df.to_csv(
    "../data/loan_cleaned.csv",
    index=False
)

In [29]:
df["Loan_Status"] = df["Loan_Status"].map({
    "Y": 1,
    "N": 0
})

In [30]:
df.drop(
    "Loan_ID",
    axis=1,
    inplace=True
)

In [31]:
X = df.drop(
    "Loan_Status",
    axis=1
)

y = df["Loan_Status"]

In [32]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [33]:
df["Loan_Status"].value_counts()

Loan_Status
1    3316
0    1684
Name: count, dtype: int64

In [34]:
categorical_cols = X.select_dtypes(
    include=["object", "string"]
).columns

numerical_cols = X.select_dtypes(
    exclude=["object", "string"]
).columns

In [35]:
categorical_transformer = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="most_frequent"
        )
    ),
    (
        "encoder",
        OneHotEncoder(
            handle_unknown="ignore"
        )
    )
])

In [36]:
numerical_transformer = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="median"
        )
    )
])

In [37]:
preprocessor = ColumnTransformer([
    (
        "cat",
        categorical_transformer,
        categorical_cols
    ),
    (
        "num",
        numerical_transformer,
        numerical_cols
    )
])

In [38]:
models = {
    "Logistic Regression":
        LogisticRegression(
            max_iter=1000
        ),

    "Random Forest":
        RandomForestClassifier(
            n_estimators=300,
            random_state=42
        ),

    "XGBoost":
        XGBClassifier(
            random_state=42,
            eval_metric="logloss"
        )
}

In [41]:
best_model = None
best_score = 0

for name, model in models.items():

    pipeline = Pipeline([
        (
            "preprocessor",
            preprocessor
        ),
        (
            "model",
            model
        )
    ])

In [44]:
for col in [
    "Gender",
    "Married",
    "Dependents",
    "Education",
    "Self_Employed",
    "Property_Area"
]:
    print("\n", col)
    print(df[col].unique())


 Gender
<ArrowStringArray>
['Female', 'Male', nan]
Length: 3, dtype: str

 Married
<ArrowStringArray>
['No', 'Yes', nan]
Length: 3, dtype: str

 Dependents
['2' '1' '3+' '0' 4 nan]

 Education
<ArrowStringArray>
['Graduate', 'Not Graduate', 'Grad']
Length: 3, dtype: str

 Self_Employed
<ArrowStringArray>
['Yes', 'No', nan]
Length: 3, dtype: str

 Property_Area
<ArrowStringArray>
['Urban', 'Rural', 'Semiurban', nan]
Length: 4, dtype: str


In [45]:
df["Dependents"] = df["Dependents"].astype(str)

In [46]:
print(df["Dependents"].unique())

<ArrowStringArray>
['2', '1', '3+', '0', '4', nan]
Length: 6, dtype: str


In [48]:
print(df["Dependents"].map(type).value_counts())

Dependents
<class 'str'>      4925
<class 'float'>      75
Name: count, dtype: int64


In [49]:
for col in categorical_cols:
    print("\n", col)
    print(df[col].map(type).value_counts())


 Gender
Gender
<class 'str'>      4923
<class 'float'>      77
Name: count, dtype: int64

 Married
Married
<class 'str'>      4934
<class 'float'>      66
Name: count, dtype: int64

 Dependents
Dependents
<class 'str'>      4925
<class 'float'>      75
Name: count, dtype: int64

 Education
Education
<class 'str'>    5000
Name: count, dtype: int64

 Self_Employed
Self_Employed
<class 'str'>      4902
<class 'float'>      98
Name: count, dtype: int64

 Property_Area
Property_Area
<class 'str'>      4936
<class 'float'>      64
Name: count, dtype: int64


In [50]:
print(categorical_cols)

print()

print(numerical_cols)

Index(['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed',
       'Property_Area'],
      dtype='str')

Index(['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount',
       'Loan_Amount_Term', 'Credit_History'],
      dtype='str')


In [52]:
for col in categorical_cols:
    print(col, df[col].dtype)

for col in numerical_cols:
    print(col, df[col].dtype)

print(df["Dependents"].unique())
print(df["Dependents"].dtype)

Gender str
Married str
Dependents str
Education str
Self_Employed str
Property_Area str
ApplicantIncome int64
CoapplicantIncome int64
LoanAmount float64
Loan_Amount_Term float64
Credit_History float64
<ArrowStringArray>
['2', '1', '3+', '0', '4', nan]
Length: 6, dtype: str
str


In [53]:
X.dtypes

Gender                   str
Married                  str
Dependents            object
Education                str
Self_Employed            str
ApplicantIncome        int64
CoapplicantIncome      int64
LoanAmount           float64
Loan_Amount_Term     float64
Credit_History       float64
Property_Area            str
dtype: object

In [54]:
for col in categorical_cols:
    X[col] = X[col].astype(object)

In [55]:
pipeline.fit(X_train, y_train)

TypeError: Encoders require their input argument must be uniformly strings or numbers. Got ['int', 'str']

In [56]:
cat_cols = [
    "Gender",
    "Married",
    "Dependents",
    "Education",
    "Self_Employed",
    "Property_Area"
]

for col in cat_cols:
    df[col] = df[col].astype(str)

In [57]:
import numpy as np

for col in cat_cols:
    df[col] = df[col].replace("nan", np.nan)

In [59]:
for col in cat_cols:
    print(col, df[col].apply(type).value_counts())

Gender Gender
<class 'str'>      4923
<class 'float'>      77
Name: count, dtype: int64
Married Married
<class 'str'>      4934
<class 'float'>      66
Name: count, dtype: int64
Dependents Dependents
<class 'str'>      4925
<class 'float'>      75
Name: count, dtype: int64
Education Education
<class 'str'>    5000
Name: count, dtype: int64
Self_Employed Self_Employed
<class 'str'>      4902
<class 'float'>      98
Name: count, dtype: int64
Property_Area Property_Area
<class 'str'>      4936
<class 'float'>      64
Name: count, dtype: int64


In [60]:
X = df.drop("Loan_Status", axis=1)
y = df["Loan_Status"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [62]:
pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](11,)","['Gender','Married','Dependents',...,'Loan_Amount_Term','Credit_History', 'Property_Area']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,11
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...), ('num', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='p

In [63]:
preds = pipeline.predict(X_test)

accuracy = accuracy_score(
    y_test,
    preds
)

print(accuracy)

0.607


In [64]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test,
        preds
    )
)

              precision    recall  f1-score   support

           0       0.37      0.23      0.28       337
           1       0.67      0.80      0.73       663

    accuracy                           0.61      1000
   macro avg       0.52      0.51      0.50      1000
weighted avg       0.57      0.61      0.58      1000



In [65]:
models = {
    "Logistic Regression":
        LogisticRegression(max_iter=1000),

    "Random Forest":
        RandomForestClassifier(
            n_estimators=300,
            random_state=42
        ),

    "XGBoost":
        XGBClassifier(
            random_state=42,
            eval_metric="logloss"
        )
}

In [66]:
df["Loan_Status"].value_counts(normalize=True)

Loan_Status
1    0.6632
0    0.3368
Name: proportion, dtype: float64

NameError: name 'logistic_score' is not defined

In [68]:
best_model = None
best_score = 0

for name, model in models.items():

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipeline.fit(X_train, y_train)

    preds = pipeline.predict(X_test)

    score = accuracy_score(y_test, preds)

    print(f"{name}: {score:.4f}")

    if score > best_score:
        best_score = score
        best_model = pipeline

print("\nBest Accuracy:", best_score)

c:\Users\prakh\OneDrive\Desktop\Projects\project 3\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Logistic Regression: 0.6630
Random Forest: 0.6620
XGBoost: 0.6070

Best Accuracy: 0.663


In [69]:
from sklearn.preprocessing import StandardScaler

In [70]:
numerical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

In [71]:
best_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(
        max_iter=5000
    ))
])

best_pipeline.fit(X_train, y_train)

joblib.dump(
    best_pipeline,
    "../models/loan_approval_pipeline.pkl"
)

['../models/loan_approval_pipeline.pkl']